# Imports

In [1]:
import numpy as np

## Manipulate path

In [2]:
import sys
from pathlib import Path
from os import chdir

In [3]:
PROJECT_DIR = str(Path(sys.path[0]).parent.parent)

In [4]:
sys.path.append(PROJECT_DIR)
chdir(PROJECT_DIR)

## WASM Imports

In [5]:
from python_src.common.wasm_handle import load_wasm_module
from python_src.common.vector_f32 import VectorF32
from python_src.common.c_array_f32 import CArrayF32
from python_src.common.byte_array import ByteArray

Create WASM runtime instance

In [6]:
STORE, INSTANCE = load_wasm_module("./emcc_wasm/build/mod.wasm")

Loading WASM module from: ./emcc_wasm/build/mod.wasm
Module imports: [('env', 'emscripten_notify_memory_growth', <wasmtime._types.FuncType object at 0x00000164FC941290>)]
Module loaded successfully. Exports: ['memory', 'add', 'multiply', 'factorial', 'power', 'create_float_vector', 'free_float_vector', 'sum_float_vector', 'set_vector_element', 'get_vector_element', 'create_float_array', 'free_float_array', 'sum_float_array', 'set_array_element', 'get_array_element', 'create_byte_array', 'free_byte_array', 'sum_byte_array', 'sum_byte_array_simd', 'set_byte_array_element', 'get_byte_array_element', '__indirect_function_table', '_initialize', '_emscripten_stack_restore', 'emscripten_stack_get_current']


**Why do we separate store and instance?**

Store is the memory associated with the WASM virtual environment. An instance is a set of functions, and registers to execute those functions. Multiple instances can use the same story and two different instances can have different permissions.

## Parameters

In [33]:
ARRAY_SIZE = 1_000_000

## Profile numpy float32 array

The baseline measurement from numpy uses pre-compiled numpy for you operating system.

In [7]:
input_float_array = np.array(range(ARRAY_SIZE), dtype=np.float32)

In [23]:
%timeit input_float_array.sum()

530 µs ± 9.25 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Profile wasm float32 vector

The 32 bit vector implementation on the WASM side.

In [14]:
vector = VectorF32(STORE, INSTANCE, ARRAY_SIZE)

In [15]:
%timeit vector.sum()

482 µs ± 23.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Profile wasm float32 C style array

The main difference is that the float32 c style array relies on Python do the boundary checks.

In [17]:
c_style_array = CArrayF32(STORE, INSTANCE, ARRAY_SIZE)

In [18]:
%timeit c_style_array.sum()

482 µs ± 13.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Profile numpy uint8 array

We do a baseline measurement for 

In [20]:
input_byte_array = np.array([1] * ARRAY_SIZE, dtype=np.uint8)

In [21]:
%timeit input_byte_array.sum()

1.06 ms ± 11.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Profile wasm byte array

With byte arrays there is a sharper increase from the optimisations from the WASM compiler.

In [25]:
byte_array = ByteArray(STORE, INSTANCE, ARRAY_SIZE)

In [29]:
%timeit byte_array.sum()

484 µs ± 6.33 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Profile wasm byte array with SIMD

We appear to get a speedup again using SIMD. This is better supported with Rust that will turn this kind of function off if it is not supported by the cpu architecture. C++ needs to provide two binaries to acheive the same behaviour.

Wasm only supports 128 bit SIMD, things like AVX512 are only supported on certain devices. The 128bit version is more of a "one size fits all" approach.

In [28]:
%timeit byte_array.sum_simd()

276 µs ± 7.83 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
